# Phase 4 : table d'analyse et indicateurs

Ce notebook part des tables nettoyées de `data/clean/` (phase 3). Il construit la table d'analyse : une ligne par transaction, avec les informations du client et de la carte, la famille du commerçant, l'étiquette de fraude et les indicateurs de comportement. Les fonctions sont dans `src/features.py`. Chaque étape est expliquée dans `docs/`.

Les indicateurs qui regardent le passé (P4-04 à P4-07) n'utilisent que les transactions précédentes : une transaction n'entre jamais dans le calcul de son propre indicateur.

## Chargement des tables nettoyées

In [1]:
import time
import pandas as pd

from src.load_data import load_clean_table
from src.features import (
    build_analysis_table, check_analysis_table, add_time_features, check_time_features,
    add_age_at_transaction, check_age, add_amount_gap, check_amount_gap,
    add_card_frequency, add_recent_auth_errors, recount_card_activity,
    add_first_time_flags, check_first_time_flags, add_context_flags, check_context_flags,
    indicator_catalogue, describe_indicators, check_missing_and_bounds, save_analysis_table,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 200)

In [2]:
df_transaction = load_clean_table("transactions")
df_users = load_clean_table("users")
df_card = load_clean_table("cards")
df_mcc = load_clean_table("mcc")
df_labels = load_clean_table("fraud_labels")
n_transactions = len(df_transaction)

for name, table in [("transactions", df_transaction), ("users", df_users), ("cards", df_card),
                    ("mcc", df_mcc), ("fraud_labels", df_labels)]:
    print(f"{name}: {table.shape[0]:,} lignes, {table.shape[1]} colonnes")

transactions: 13,305,915 lignes, 30 colonnes
users: 2,000 lignes, 14 colonnes
cards: 6,146 lignes, 11 colonnes
mcc: 109 lignes, 4 colonnes
fraud_labels: 8,914,963 lignes, 2 colonnes


## P4-01 : construire la table d'analyse

Les transactions reçoivent les colonnes utiles des clients, des cartes, la famille du code MCC et l'étiquette de fraude. Les colonnes brutes `merchant_state` et `zip` sont retirées : depuis P3-07, elles sont remplacées par `merchant_state_clean`, `merchant_country` et `zip_clean`.

Critère de fin : même nombre de lignes que la table des transactions. Voir `docs/P4-01_table_analyse.md`.

In [3]:
df = build_analysis_table(df_transaction, df_users, df_card, df_mcc, df_labels)
del df_transaction, df_labels

controle = check_analysis_table(df, n_transactions)
print("Critère de fin validé :", controle["statut"].all())
controle

Critère de fin validé : True


,contrôle,obtenu,statut
0,lignes de la table d'analyse,13305915,True
1,identifiants en double,0,True
2,valeurs vides dans les colonnes ajoutées,0,True
3,transactions étiquetées (is_fraud),8914963,True


In [4]:
print(f"Table d'analyse : {df.shape[0]:,} lignes, {df.shape[1]} colonnes, "
      f"{df.memory_usage(deep=True).sum() / 1e6:,.1f} Mo en mémoire")
df[["id", "date", "client_id", "card_id", "amount", "gender", "card_brand", "card_type",
    "mcc_family", "is_fraud"]].head()

Table d'analyse : 13,305,915 lignes, 44 colonnes, 1,334.3 Mo en mémoire


,id,date,client_id,card_id,amount,gender,card_brand,card_type,mcc_family,is_fraud
0,7475327,2010-01-01 00:01:00,1556,2972,-77.00,Female,Mastercard,Debit (Prepaid),Alimentation,0
1,7475328,2010-01-01 00:02:00,561,4575,14.57,Male,Mastercard,Credit,Grandes surfaces et commerces divers,0
2,7475329,2010-01-01 00:02:00,1129,102,80.00,Male,Mastercard,Debit,Services professionnels et financiers,0
3,7475331,2010-01-01 00:05:00,430,2860,200.00,Female,Mastercard,Debit,Services professionnels et financiers,<NA>
4,7475332,2010-01-01 00:06:00,848,3915,46.41,Male,Visa,Debit,Restauration et bars,0


**Constats**

- La table d'analyse a 13 305 915 lignes, comme la table des transactions : le critère de fin est validé.
- Aucun identifiant n'est en double et aucune colonne ajoutée n'est vide.
- 8 914 963 transactions ont une étiquette de fraude ; les autres ont une valeur vide dans `is_fraud`.
- La table a 44 colonnes et occupe 1 334,3 Mo en mémoire à cette étape.

## P4-02 : variables de temps

Année, mois, jour de la semaine (0 = lundi), heure, et deux indicateurs : `is_night` (de 0 h à 5 h 59) et `is_weekend` (samedi et dimanche).

Critère de fin : colonnes vérifiées sur un échantillon. Voir `docs/P4-02_variables_temps.md`.

In [5]:
df = add_time_features(df)

verif = check_time_features(df)
print("Critère de fin validé :", verif["identique au recalcul"].all())
verif

Critère de fin validé : True


,id,date,year,month,weekday,hour,is_night,is_weekend,identique au recalcul
12119134,22293358,2018-12-25 06:11:00,2018,12,1,6,0,0,True
11911763,22036862,2018-10-31 15:17:00,2018,10,2,15,0,0,True
11922349,22049988,2018-11-03 10:32:00,2018,11,5,10,0,1,True
5685601,14384899,2014-05-09 11:16:00,2014,5,4,11,0,0,True
706192,8320954,2010-07-30 10:15:00,2010,7,4,10,0,0,True
2047504,9947306,2011-08-18 10:17:00,2011,8,3,10,0,0,True
7068918,16081229,2015-05-12 16:51:00,2015,5,1,16,0,0,True
8054695,17289178,2016-01-25 22:31:00,2016,1,0,22,0,0,True
3933095,12245170,2013-01-22 20:16:00,2013,1,1,20,0,0,True
10050121,19741094,2017-07-02 06:58:00,2017,7,6,6,0,1,True


In [6]:
pd.Series({
    "transactions de nuit (0 h à 5 h 59)": int(df["is_night"].sum()),
    "transactions le week-end": int(df["is_weekend"].sum()),
    "transactions au total": len(df),
})

transactions de nuit (0 h à 5 h 59)      770383
transactions le week-end                3801414
transactions au total                  13305915
dtype: int64

**Constats**

- Les six variables de temps sont identiques au recalcul ligne par ligne sur les 10 transactions de l'échantillon.
- 770 383 transactions ont lieu la nuit (de 0 h à 5 h 59) et 3 801 414 le week-end, sur 13 305 915.

## P4-03 : âge du client à la date de la transaction

L'âge est calculé avec `birth_year` et `birth_month`, et non avec `current_age`, qui est l'âge en février 2020 (P2-12). Le jour de naissance n'est pas connu : l'anniversaire compte à partir du premier jour du mois de naissance.

Critère de fin : âges vérifiés à la main sur 10 transactions. Voir `docs/P4-03_age_transaction.md`.

In [7]:
df = add_age_at_transaction(df)

verif = check_age(df)
print("Critère de fin validé :", verif["identique"].all())
verif

Critère de fin validé : True


,id,date,birth_year,birth_month,age_at_transaction,calcul à la main,identique
475161,8044167,2010-05-23 17:06:00,1971,11,38,2010 - 1971 - 1 = 38,True
956503,8622107,2010-10-10 17:00:00,1987,11,22,2010 - 1987 - 1 = 22,True
1947052,9825302,2011-07-21 07:34:00,1967,5,44,2011 - 1967 - 0 = 44,True
4466474,12896682,2013-06-16 15:05:00,1939,1,74,2013 - 1939 - 0 = 74,True
9827373,19465911,2017-05-05 06:32:00,1961,7,55,2017 - 1961 - 1 = 55,True
1695934,9520089,2011-05-11 08:21:00,1982,3,29,2011 - 1982 - 0 = 29,True
932271,8592854,2010-10-03 16:26:00,1967,2,43,2010 - 1967 - 0 = 43,True
13159258,23580546,2019-09-23 09:52:00,1965,10,53,2019 - 1965 - 1 = 53,True
10060816,19754317,2017-07-04 21:29:00,1994,2,23,2017 - 1994 - 0 = 23,True
12380611,22617370,2019-03-03 14:41:00,1957,8,61,2019 - 1957 - 1 = 61,True


In [8]:
ecart_age = (df_users.set_index("id")["current_age"].reindex(df["client_id"]).to_numpy()
             - df["age_at_transaction"].to_numpy())
print(f"Âge à la transaction : de {df['age_at_transaction'].min()} à {df['age_at_transaction'].max()} ans")
print(f"Écart avec current_age : de {ecart_age.min()} à {ecart_age.max()} ans")

Âge à la transaction : de 17 à 101 ans
Écart avec current_age : de 0 à 11 ans


**Constats**

- Les 10 âges calculés sont identiques au calcul à la main.
- L'âge à la transaction va de 17 à 101 ans.
- L'écart avec `current_age` va de 0 à 11 ans : utiliser `current_age` aurait vieilli les clients jusqu'à 11 ans pour les transactions de 2010.

## P4-04 : écart au montant habituel du client

Le montant habituel est la médiane des achats précédents du client (montants positifs seulement). Il faut au moins 10 achats précédents : avant, la colonne reste vide. `amount_gap` est l'écart en dollars et `amount_ratio` le rapport (2 veut dire deux fois le montant habituel).

Critère de fin : colonne vérifiée sur un client. Voir `docs/P4-04_ecart_montant_habituel.md`.

In [9]:
debut = time.perf_counter()
df = add_amount_gap(df)
print(f"Calcul : {time.perf_counter() - debut:.0f} s")

nb_tx_client = df["client_id"].value_counts()
client_exemple = int(nb_tx_client.index[len(nb_tx_client) // 2])
verif = check_amount_gap(df, client_exemple)
verif

Calcul : 37 s
Client 363 : 9832 transactions, toutes identiques au recalcul : True


,id,date,amount,achats précédents,médiane recalculée,client_median_amount,identique
8,7490589,2010-01-04 20:23:00,4.14,8,NaN,NaN,True
9,7491049,2010-01-05 04:09:00,30.22,9,NaN,NaN,True
10,7495069,2010-01-06 04:26:00,32.82,10,9.28,9.28,True
11,7497994,2010-01-06 16:23:00,13.08,11,10.40,10.40,True
12,7498272,2010-01-06 17:51:00,23.97,12,11.03,11.03,True
13,7500188,2010-01-07 09:53:00,17.38,13,11.66,11.66,True
14,7500445,2010-01-07 10:50:00,12.40,14,11.80,11.80,True
15,7502560,2010-01-07 20:11:00,4.57,15,11.94,11.94,True
16,7503932,2010-01-08 09:59:00,13.01,16,11.80,11.80,True
17,7503990,2010-01-08 10:16:00,13.80,17,11.94,11.94,True


In [10]:
ratio = df.loc[df["amount"] > 0, "amount_ratio"]
pd.Series({
    "transactions sans montant habituel": int(df["client_median_amount"].isna().sum()),
    "achats avec un rapport calculé": int(ratio.notna().sum()),
    "rapport médian": round(float(ratio.median()), 2),
    "rapport au quantile 99 %": round(float(ratio.quantile(0.99)), 2),
    "rapport au quantile 99,9 %": round(float(ratio.quantile(0.999)), 2),
    "achats à 10 fois le montant habituel ou plus": int((ratio >= 10).sum()),
})

transactions sans montant habituel                 12787.00
achats avec un rapport calculé                  12623037.00
rapport médian                                         1.00
rapport au quantile 99 %                              16.85
rapport au quantile 99,9 %                            61.63
achats à 10 fois le montant habituel ou plus      305792.00
dtype: float64

**Constats**

- Pour le client 363 (9 832 transactions), le montant habituel recalculé avec une boucle simple est identique pour toutes les transactions : le critère de fin est validé.
- Le calcul sur toute la table prend 37 s.
- 12 787 transactions n'ont pas de montant habituel : leur client avait moins de 10 achats précédents.
- Sur 12 623 037 achats avec un rapport calculé, le rapport médian est 1,00 : un achat typique est au niveau du montant habituel.
- Le rapport vaut 16,85 au quantile 99 % et 61,63 au quantile 99,9 %. 305 792 achats sont au moins 10 fois plus élevés que le montant habituel.

## P4-05 : fréquence d'utilisation de la carte

`card_tx_1h` et `card_tx_24h` comptent les transactions précédentes de la même carte dans l'heure et dans les 24 heures.

Critère de fin : colonnes vérifiées sur une carte. Voir `docs/P4-05_frequence_carte.md`.

In [11]:
debut = time.perf_counter()
df = add_card_frequency(df)
print(f"Calcul : {time.perf_counter() - debut:.0f} s")

# Carte de vérification : la plus grande rafale parmi les cartes de moins de 5 000 transactions
nb_tx_carte = df["card_id"].value_counts()
petites_cartes = nb_tx_carte[nb_tx_carte <= 5000].index
max_1h = df.loc[df["card_id"].isin(petites_cartes)].groupby("card_id")["card_tx_1h"].max()
carte_rafale = int(max_1h.idxmax())

verif = recount_card_activity(df, carte_rafale)
print(f"Carte {carte_rafale} : {len(verif)} transactions, "
      f"toutes identiques au recompte : {verif['identique_frequence'].all()}")
verif.loc[verif["card_tx_1h"] > 0, ["id", "date", "card_tx_1h", "recompte_1h", "card_tx_24h",
                                     "recompte_24h", "identique_frequence"]].head(10)

Calcul : 32 s
Carte 2048 : 4693 transactions, toutes identiques au recompte : True


,id,date,card_tx_1h,recompte_1h,card_tx_24h,recompte_24h,identique_frequence
12,7571051,2010-01-25 13:36:00,1,1,2,2,True
13,7571121,2010-01-25 13:51:00,2,2,3,3,True
16,7585051,2010-01-29 04:55:00,1,1,1,1,True
19,7607096,2010-02-03 15:15:00,1,1,1,1,True
21,7646470,2010-02-13 14:04:00,1,1,1,1,True
22,7646525,2010-02-13 14:16:00,2,2,2,2,True
53,7974479,2010-05-06 13:41:00,1,1,2,2,True
54,7974546,2010-05-06 13:58:00,2,2,3,3,True
66,8044802,2010-05-23 21:58:00,1,1,1,1,True
97,8309655,2010-07-27 13:41:00,1,1,2,2,True


In [12]:
df[["card_tx_1h", "card_tx_24h"]].describe(percentiles=[0.5, 0.99, 0.999]).round(2)

,card_tx_1h,card_tx_24h
count,13305915.00,13305915.00
mean,0.28,1.91
std,0.66,2.03
min,0.00,0.00
50%,0.00,1.00
99%,3.00,9.00
99.9%,5.00,14.00
max,12.00,30.00


**Constats**

- Pour la carte 2048 (4 693 transactions), les deux compteurs sont identiques au recompte fait transaction par transaction.
- Le calcul sur toute la table prend 32 s.
- La médiane de `card_tx_1h` est 0 et celle de `card_tx_24h` est 1.
- Au quantile 99,9 %, une carte compte 5 transactions dans l'heure précédente et 14 dans les 24 heures. Les maximums sont 12 et 30.